
<div style="border-radius:10px; border:1px solid #a3a3a3; padding:24px; background:linear-gradient(120deg, #ffffff 80%, #e7effc 100%); box-shadow:0 2px 12px 0 rgba(40,70,140,.07);">

<h2 style="color:#226db0; margin-top:0;">
  🔁 SLOWLY CHANGING DIMENSION TYPE 2 (SCD)
</h2>

<p style="font-size:1.13em; color:#233140; line-height:1.6em;">
  The <strong>NYCTAXI.SILVER.REGIONS_SCD_2</strong> table has been loaded using the SCD Type 2 methodology, ensuring auditability of region changes. Every update preserves historical data with <code>effective_start_date</code> and <code>effective_end_date</code> boundaries.
</p>

<ul style="font-size:1.04em; color:#333c4f;">
  <li><strong>Deduplication</strong> performed to ensure one entry per <code>id</code>, keeping the most recent.</li>
  <li><strong>Change detection</strong> identifies new or modified regions.</li>
  <li><strong>Automatic versioning</strong>: Previous records expired; new rows appended with updated data and timestamps.</li>
  <li>Each region’s lifecycle is transparently tracked for auditing and analytics.</li>
</ul>

<p style="margin-top:24px; color:#1b4475;"> 
  <em>Historical trends are never lost—every change in your regions is captured and retained.</em>
</p>

</div>

In [0]:
from pyspark.sql.functions import *

changed_regions_df = (spark.read.format('csv')
                        .option('header', 'true')
						.schema('id INT, name STRING, date STRING')
                        .load('/Volumes/nyctaxi/landing/yellow_taxi/regions/')
			).withColumn('load_timestamp', current_timestamp())

changed_regions_df = changed_regions_df.withColumn(
    "date_ts", to_timestamp(col("date"), "dd-MM-yyyy")
)

changed_regions_df.orderBy('id').display()

In [0]:
# =====================================================
# IMPORTS
# =====================================================

from delta.tables import DeltaTable
from datetime import datetime

from pyspark.sql.window import Window

from pyspark.sql.functions import (
    col,
    lit,
    current_timestamp,
    to_timestamp,
    row_number,
    regexp_extract,
    round
)

from pyspark.sql.types import (
    TimestampType,
    DecimalType
)

# =====================================================
# LOAD START TIME
# =====================================================

load_start_time = datetime.now()

# =====================================================
# WIDGET PARAMETERS
# =====================================================

dbutils.widgets.text("log_id", "")
dbutils.widgets.text("run_id", "")
dbutils.widgets.text("event_time", "")
dbutils.widgets.text("event_type", "")
dbutils.widgets.text("source_table", "")
dbutils.widgets.text("target_table", "")
dbutils.widgets.text("layer", "")
dbutils.widgets.text("status", "")
dbutils.widgets.text("message", "")
dbutils.widgets.text("notebook_path", "")
dbutils.widgets.text("pipeline_name", "")

log_id = dbutils.widgets.get("log_id")
run_id = dbutils.widgets.get("run_id")
raw_event_time = dbutils.widgets.get("event_time")

source_table = dbutils.widgets.get("source_table")
target_table = dbutils.widgets.get("target_table")
layer = dbutils.widgets.get("layer")

notebook_path = dbutils.widgets.get("notebook_path")
pipeline_name = dbutils.widgets.get("pipeline_name")

# =====================================================
# EVENT DATE
# =====================================================

try:
    if (
        not raw_event_time
        or raw_event_time.startswith("{{")
    ):
        event_time = datetime.now().date()
    else:
        event_time = datetime.fromisoformat(
            raw_event_time
        ).date()
except:
    event_time = datetime.now().date()

# =====================================================
# DEFAULT VALUES
# =====================================================

status = "SUCCESS"
event_type = "LOAD_SUCCESS"
message = ""

source_record_count = 0
deduped_record_count = 0
changed_record_count = 0
updated_record_count = 0
inserted_record_count = 0

# =====================================================
# SCD TYPE 2 PROCESS
# =====================================================

try:

    # ==============================================
    # SOURCE PREPARATION
    # ==============================================

    source_df = changed_regions_df.withColumn("date_ts",to_timestamp(col("date"),"dd-MM-yyyy"))

    source_record_count = source_df.count()

    # ==============================================
    # DEDUPLICATE SOURCE
    # ==============================================

    w = Window.partitionBy("id").orderBy(
        col("date_ts").desc()
    )

    deduped_source = (
        source_df
        .withColumn(
            "rn",
            row_number().over(w)
        )
        .filter(col("rn") == 1)
        .drop("rn")
    )

    deduped_record_count = deduped_source.count()

    # ==============================================
    # LOAD TARGET TABLE
    # ==============================================

    dt = DeltaTable.forName(
        spark,
        "NYCTAXI.SILVER.REGIONS_SCD_2"
    )

    current_active = (
        dt.toDF()
        .filter(
            col("effective_end_date").isNull()
        )
    )

    # ==============================================
    # IDENTIFY CHANGES
    # ==============================================

    changed_ids = (
        deduped_source.alias("s")
        .join(
            current_active.alias("t"),
            col("s.id") == col("t.id"),
            "left"
        )
        .filter(
            (col("t.id").isNull())
            |
            (
                ~col("s.name")
                .eqNullSafe(col("t.name"))
            )
        )
        .select(
            col("s.id").alias("id")
        )
        .distinct()
        .cache()
    )

    changed_record_count = changed_ids.count()

    # ==============================================
    # UPDATED IDS
    # ==============================================

    updated_ids = (
        deduped_source.alias("s")
        .join(
            current_active.alias("t"),
            col("s.id") == col("t.id"),
            "inner"
        )
        .filter(
            ~col("s.name")
            .eqNullSafe(col("t.name"))
        )
        .select(
            col("s.id")
        )
        .distinct()
        .cache()
    )

    updated_record_count = updated_ids.count()

    # ==============================================
    # EXPIRE EXISTING ACTIVE RECORDS
    # ==============================================

    dt.alias("t").merge(
        updated_ids.alias("u"),
        """
        t.id = u.id
        AND t.effective_end_date IS NULL
        """
    ).whenMatchedUpdate(
        set={
            "effective_end_date":
                current_timestamp()
        }
    ).execute()

    # ==============================================
    # INSERT NEW RECORD VERSIONS
    # ==============================================

    new_versions = (
        deduped_source.alias("s")
        .join(
            changed_ids.alias("c"),
            col("s.id") == col("c.id"),
            "inner"
        )
        .select(
            col("s.id").alias("id"),
            col("s.name").alias("name"),
            col("s.date_ts")
                .alias(
                    "effective_start_date"
                )
        )
        .withColumn(
            "effective_end_date",
            lit(None).cast(
                TimestampType()
            )
        )
        .withColumn(
            "load_timestamp",
            current_timestamp()
        )
    )

    inserted_record_count = new_versions.count()

    new_versions.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable(
            "NYCTAXI.SILVER.REGIONS_SCD_2"
        )

    message = (
        f"SCD2 Successful | "
        f"Source={source_record_count} | "
        f"Deduped={deduped_record_count} | "
        f"Changed={changed_record_count} | "
        f"Updated={updated_record_count} | "
        f"Inserted={inserted_record_count}"
    )

except Exception as e:

    status = "FAILED"
    event_type = "LOAD_FAILURE"
    message = str(e)[:4000]

# =====================================================
# LOAD END TIME
# =====================================================

load_end_time = datetime.now()

execution_duration_seconds = (
    load_end_time - load_start_time
).total_seconds()

# =====================================================
# CURRENT USER
# =====================================================

user_name = spark.sql(
    "SELECT current_user()"
).first()[0]

# =====================================================
# FILE METADATA
# =====================================================

file_metadata_df = (
        changed_regions_df.select(col("_metadata.file_name").alias("file_name"),
                           col("_metadata.file_path").alias("file_path"),
                           col("_metadata.file_size").cast("bigint").alias("file_size_bytes"),
                           col("_metadata.file_modification_time").alias("file_modified_time")
    ).distinct()
)

file_metadata_df = (
    file_metadata_df
    .withColumn("file_extension",regexp_extract(col("file_name"),r"\.([^\.]+)$",1))
    .withColumn("source_system",lit("NYCTAXI"))
    .withColumn("source_folder",regexp_extract(col("file_path"),r".*/([^/]+)/[^/]+$",1))
    .withColumn("file_size_mb",round(col("file_size_bytes")/ (1024 * 1024),2).cast(DecimalType(18,2)))
    .withColumn("file_created_time",col("file_modified_time")))

# =====================================================
# BUILD AUDIT DATAFRAME
# =====================================================

audit_df = (
    file_metadata_df
    .withColumn("log_id", lit(log_id))
    .withColumn("run_id", lit(run_id))
    .withColumn("event_time", lit(event_time).cast("date"))
    .withColumn("event_type", lit(event_type))
    .withColumn("source_table", lit(source_table))
    .withColumn("target_table", lit(target_table))
    .withColumn("layer", lit(layer))
    .withColumn("record_count", lit(inserted_record_count))
    .withColumn("status", lit(status))
    .withColumn("message", lit(message))
    .withColumn("user_name", lit(user_name))
    .withColumn("notebook_path", lit(notebook_path))
    .withColumn("pipeline_name", lit(pipeline_name))
    .withColumn("load_start_time", lit(load_start_time).cast("timestamp"))
    .withColumn("load_end_time", lit(load_end_time).cast("timestamp"))
)

# =====================================================
# WRITE AUDIT LOG
# =====================================================

audit_df.write.format("delta").mode("append").saveAsTable("NYCTAXI.AUDIT.PIPELINE_EXECUTION_LOG")

# =====================================================
# FINAL OUTPUT
# =====================================================
print("=" * 60)
print(f"Run ID : {run_id}")
print(f"Status : {status}")
print(f"Changed : {changed_record_count}")
print(f"Updated : {updated_record_count}")
print(f"Inserted : {inserted_record_count}")
print(f"Duration : {execution_duration_seconds} sec")
print("=" * 60)

In [0]:
dbutils.notebook.exit('SLOWLY CHANGINNG DIMENSION TYPE 2 IS SUCCESSFULLY LOADED INTO NYCTAXI.SILVER.REGIONS_SCD_2')